# HDB Resale Flat Price ETL PipelineComplete ETL for HDB resale flat prices (Jan 2012 to Dec 2016).**Table of Contents**1. [Setup](#section-1)2. [Load & Combine](#section-2)3. [Data Profiling](#section-3)4. [Data Validation](#section-4)5. [Remaining Lease](#section-5)6. [Composite-Key Deduplication](#section-6)7. [Anomaly Detection](#section-7)8. [Additional Cleaning](#section-8)9. [Resale Identifier](#section-9)10. [Hash Identifier](#section-10)11. [Assemble Failed Dataset](#section-11)12. [Save Outputs](#section-12)13. [Pipeline Summary](#section-13)

<a id="section-1"></a>## 1. SetupImport libraries and define project paths.

In [ ]:
import hashlib
import shutil
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

_PROJECT_ROOT = Path(
    r"C:\Users\nghosh\OneDrive\docs\kaajer\staff-aug\HDB\DE-2026-07\QL\Part_1"
)
RAW_DIR = _PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = _PROJECT_ROOT / "data" / "processed"
RAW_OUT_DIR = PROCESSED_DIR / "raw"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
RAW_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Paths configured:")
print(f"  RAW_DIR: {RAW_DIR}")
print(f"  PROCESSED_DIR: {PROCESSED_DIR}")
print(f"  RAW_OUT_DIR: {RAW_OUT_DIR}")

<a id="section-2"></a>## 2. Load & CombineRead all raw CSVs into a single master dataset.- Filter 2000 - Feb 2012 file to rows from Jan 2012 onward.- The Jan 2015 - Dec 2016 file contains an extra column `remaining_lease`.  The union preserves all attributes from every file.

In [ ]:
csv_files = sorted(RAW_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files:")
for _f in csv_files:
    print(f"  - {_f.name}")

_dfs = []
for _csv_file in csv_files:
    _df = pd.read_csv(_csv_file)
    _original_count = len(_df)
    if "2000 - Feb 2012" in _csv_file.name:
        _df = _df[_df["month"] >= "2012-01"].copy()
        print(
            f"Loaded {_csv_file.name}: {_original_count} raw, "
            f"{len(_df)} after Jan 2012 filter"
        )
    else:
        print(f"Loaded {_csv_file.name}: {len(_df)} rows")
    _dfs.append(_df)

combined_df = pd.concat(_dfs, ignore_index=True, sort=False)
print(f"Records combined: {len(combined_df)}")
_cols = list(combined_df.columns)
print(f"Columns ({len(_cols)}): {_cols}")

for _src in csv_files:
    _dst = RAW_OUT_DIR / _src.name
    shutil.copy2(_src, _dst)
print(f"Raw files copied to: {RAW_OUT_DIR}")

<a id="section-3"></a>## 3. Data ProfilingStatistical overview of the master dataset to inform validation-rule design.

In [ ]:
print("=== MASTER DATASET PROFILE ===")
print(f"Shape: {combined_df.shape}")
print("Columns and types:")
print(combined_df.dtypes)

print("Numeric summary:")
_numeric_cols = (
    combined_df
    .select_dtypes(include=[np.number])
    .columns
    .tolist()
)
print(combined_df[_numeric_cols].describe())

print("Categorical summary:")
_cat_cols = ["town", "flat_type", "flat_model", "storey_range"]
for _c in _cat_cols:
    _n = combined_df[_c].nunique()
    print(f"{_c.upper()} - {_n} unique values:")
    print(combined_df[_c].value_counts().head(10))

print("Date range:")
_min_month = combined_df["month"].min()
_max_month = combined_df["month"].max()
print(f"  {_min_month} to {_max_month}")

print("Missing values:")
print(combined_df.isnull().sum())

<a id="section-4"></a>## 4. Data ValidationValidation rules are derived from the statistical properties of the master dataset, not hardcoded arbitrary ranges.| Field | Rule ||-------|------|| month | Within observed [min_month, max_month] || town | Must be a known town in the dataset || flat_type | Must be a known flat type in the dataset || flat_model | Must be a known model in the dataset || storey_range | Must be a known range in the dataset || resale_price | Non-negative and within Q1 +/- 3*IQR || floor_area_sqm | Positive and within Q1 +/- 3*IQR |

In [ ]:
_MIN_MONTH = combined_df["month"].min()
_MAX_MONTH = combined_df["month"].max()
_VALID_TOWNS = set(combined_df["town"].dropna().unique())
_VALID_FLAT_TYPES = set(
    combined_df["flat_type"].dropna().unique()
)
_VALID_FLAT_MODELS = set(
    combined_df["flat_model"].dropna().unique()
)
_VALID_STOREY_RANGES = set(
    combined_df["storey_range"].dropna().unique()
)

# Price IQR bounds
_P_Q1 = combined_df["resale_price"].quantile(0.25)
_P_Q3 = combined_df["resale_price"].quantile(0.75)
_P_IQR = _P_Q3 - _P_Q1
_P_LOWER = _P_Q1 - 3 * _P_IQR
_P_UPPER = _P_Q3 + 3 * _P_IQR

# Floor area IQR bounds
_A_Q1 = combined_df["floor_area_sqm"].quantile(0.25)
_A_Q3 = combined_df["floor_area_sqm"].quantile(0.75)
_A_IQR = _A_Q3 - _A_Q1
_A_LOWER = _A_Q1 - 3 * _A_IQR
_A_UPPER = _A_Q3 + 3 * _A_IQR

print("Validation rules derived from master dataset:")
print(f"  Date range: {_MIN_MONTH} to {_MAX_MONTH}")
print(f"  Towns: {len(_VALID_TOWNS)} known values")
print(f"  Flat types: {len(_VALID_FLAT_TYPES)} known values")
print(f"  Flat models: {len(_VALID_FLAT_MODELS)} known values")
print(f"  Storey ranges: {len(_VALID_STOREY_RANGES)} known values")
print(f"  Price bounds: {_P_LOWER:.0f} to {_P_UPPER:.0f}")
print(f"  Floor area bounds: {_A_LOWER:.1f} to {_A_UPPER:.1f}")


def validate_row(row):
    """
    Validate a single record against statistical rules.
    """
    _errors = []

    _month = row.get("month")
    if pd.isna(_month):
        _errors.append("missing_month")
    elif not (_MIN_MONTH <= str(_month) <= _MAX_MONTH):
        _errors.append(f"month_out_of_range ({_month})")

    _town = row.get("town")
    if pd.isna(_town):
        _errors.append("missing_town")
    elif str(_town).strip().upper() not in {
        t.upper() for t in _VALID_TOWNS
    }:
        _errors.append(f"invalid_town ({_town})")

    _ft = row.get("flat_type")
    if pd.isna(_ft):
        _errors.append("missing_flat_type")
    elif str(_ft).strip().upper() not in {
        t.upper() for t in _VALID_FLAT_TYPES
    }:
        _errors.append(f"invalid_flat_type ({_ft})")

    _fm = row.get("flat_model")
    if pd.isna(_fm):
        _errors.append("missing_flat_model")
    elif str(_fm).strip().upper() not in {
        m.upper() for m in _VALID_FLAT_MODELS
    }:
        _errors.append(f"invalid_flat_model ({_fm})")

    _sr = row.get("storey_range")
    if pd.isna(_sr):
        _errors.append("missing_storey_range")
    elif str(_sr).strip().upper() not in {
        s.upper() for s in _VALID_STOREY_RANGES
    }:
        _errors.append(f"invalid_storey_range ({_sr})")

    _price = row.get("resale_price")
    if pd.isna(_price):
        _errors.append("missing_resale_price")
    else:
        try:
            _p = float(_price)
            if _p < 0:
                _errors.append("negative_resale_price")
            elif _p < _P_LOWER or _p > _P_UPPER:
                _errors.append(f"price_outlier ({_p:.0f})")
        except (ValueError, TypeError):
            _errors.append(f"non_numeric_price ({_price})")

    _area = row.get("floor_area_sqm")
    if pd.isna(_area):
        _errors.append("missing_floor_area_sqm")
    else:
        try:
            _a = float(_area)
            if _a <= 0:
                _errors.append("non_positive_floor_area")
            elif _a < _A_LOWER or _a > _A_UPPER:
                _errors.append(f"area_outlier ({_a:.1f})")
        except (ValueError, TypeError):
            _errors.append(f"non_numeric_area ({_area})")

    _lcd = row.get("lease_commence_date")
    if pd.isna(_lcd):
        _errors.append("missing_lease_commence_date")
    else:
        try:
            _y = int(_lcd)
            if _y < 1900 or _y > 2050:
                _errors.append(f"lcd_out_of_range ({_y})")
        except (ValueError, TypeError):
            _errors.append(f"non_numeric_lcd ({_lcd})")

    return "; ".join(_errors) if _errors else None


combined_df["validation_error"] = (
    combined_df.apply(validate_row, axis=1)
)
_valid_mask = combined_df["validation_error"].isna()

df_valid = combined_df[_valid_mask].copy()
df_failed_validation = combined_df[~_valid_mask].copy()

print(f"Validation results:")
print(f"  Valid records: {len(df_valid)}")
print(f"  Failed records: {len(df_failed_validation)}")
print(f"Top failure reasons:")
print(
    df_failed_validation
    ["validation_error"]
    .value_counts()
    .head(10)
)

<a id="section-5"></a>## 5. Remaining LeaseAssume HDB lease is 99 years from lease_commence_date.Compute remaining lease as of today and round down to Years and Months.Formula: remaining_lease = 99 years - elapsed time since lease_commence. Rounded down.

In [ ]:
_TODAY = datetime.now()


def compute_remaining_lease(lease_commence_year):
    """
    Compute remaining lease in years and months.
    """
    if pd.isna(lease_commence_year):
        return None, None
    try:
        _y = int(lease_commence_year)
    except (ValueError, TypeError):
        return None, None

    _months_elapsed = (_TODAY.year - _y) * 12 + (
        _TODAY.month - 1
    )
    _total_lease_months = 99 * 12
    _remaining = _total_lease_months - _months_elapsed

    if _remaining < 0:
        _remaining = 0

    _rem_years = _remaining // 12
    _rem_months = _remaining % 12
    return _rem_years, _rem_months


_remaining = df_valid["lease_commence_date"].apply(
    compute_remaining_lease
)

df_valid = df_valid.copy()
df_valid["remaining_lease_years"] = [
    _r[0] for _r in _remaining
]
df_valid["remaining_lease_months"] = [
    _r[1] for _r in _remaining
]


def _format_lease(row):
    """
    Format remaining lease as a human-readable string.
    """
    _y = row.get("remaining_lease_years")
    _m = row.get("remaining_lease_months")
    if pd.isna(_y) or pd.isna(_m):
        return None
    return f"{int(_y)} years {int(_m)} months"


df_valid["remaining_lease"] = df_valid.apply(
    _format_lease,
    axis=1,
)

_ok_count = df_valid["remaining_lease"].notna().sum()
print(f"Remaining lease recomputed for {_ok_count} records.")
print("Sample:")
print(
    df_valid[
        [
            "month",
            "town",
            "lease_commence_date",
            "remaining_lease",
        ]
    ]
    .head()
)

<a id="section-6"></a>## 6. Composite-Key DeduplicationComposite key = all columns except resale_price.If duplicate keys exist, keep the record with the higher price and discard the lower-price record(s) into the Failed dataset.

In [ ]:
_EXCLUDE_KEY = [
    "resale_price",
    "validation_error",
    "remaining_lease_years",
    "remaining_lease_months",
    "remaining_lease",
]
_KEY_COLS = [
    _c for _c in df_valid.columns
    if _c not in _EXCLUDE_KEY
]

print(f"Key columns ({len(_KEY_COLS)}):")
print(f"  {_KEY_COLS}")

df_sorted = df_valid.sort_values(
    by=_KEY_COLS + ["resale_price"],
    ascending=[True] * len(_KEY_COLS) + [False],
)

_dedup_mask = ~df_sorted.duplicated(
    subset=_KEY_COLS,
    keep="first",
)

df_deduped = df_sorted[_dedup_mask].copy()
df_dup_discarded = df_sorted[~_dedup_mask].copy()
df_dup_discarded["validation_error"] = (
    "duplicate_composite_key_lower_price"
)

print(f"Records after dedup: {len(df_deduped)}")
print(f"Duplicates discarded: {len(df_dup_discarded)}")

<a id="section-7"></a>## 7. Anomaly Detection**Heuristic:** Grouped IQR on resale_price per (town, flat_type).**Assumptions:**- Price varies significantly across different towns and flat types.- A global threshold would be too loose for small flats or too strict for large ones.- For each (town, flat_type) group, prices outside Q1 - 1.5*IQR to Q3 + 1.5*IQR are flagged.- Groups with fewer than 5 transactions are excluded.

In [ ]:
df_anomaly = df_deduped.copy()

_group_stats = (
    df_anomaly
    .groupby(["town", "flat_type"])
    ["resale_price"]
    .agg(
        q1=lambda _x: _x.quantile(0.25),
        q3=lambda _x: _x.quantile(0.75),
        count="count",
    )
    .reset_index()
)

_group_stats["iqr"] = (
    _group_stats["q3"] - _group_stats["q1"]
)
_group_stats["lower_bound"] = (
    _group_stats["q1"] - 1.5 * _group_stats["iqr"]
)
_group_stats["upper_bound"] = (
    _group_stats["q3"] + 1.5 * _group_stats["iqr"]
)

df_anomaly = df_anomaly.merge(
    _group_stats[
        [
            "town",
            "flat_type",
            "lower_bound",
            "upper_bound",
            "count",
        ]
    ],
    on=["town", "flat_type"],
    how="left",
)

df_anomaly["anomaly"] = False
df_anomaly.loc[
    (
        df_anomaly["count"] >= 5
    ) & (
        (
            df_anomaly["resale_price"]
            < df_anomaly["lower_bound"]
        ) | (
            df_anomaly["resale_price"]
            > df_anomaly["upper_bound"]
        )
    ),
    "anomaly",
] = True

_anomaly_count = df_anomaly["anomaly"].sum()
print(f"Anomalous records detected: {_anomaly_count}")

df_cleaned = (
    df_anomaly[~df_anomaly["anomaly"]]
    .drop(
        columns=[
            "lower_bound",
            "upper_bound",
            "count",
            "anomaly",
        ]
    )
    .copy()
)

df_anomaly_failed = (
    df_anomaly[df_anomaly["anomaly"]]
    .copy()
)
df_anomaly_failed["validation_error"] = (
    "anomalous_resale_price_by_grouped_iqr"
)
df_anomaly_failed = (
    df_anomaly_failed
    .drop(
        columns=[
            "lower_bound",
            "upper_bound",
            "count",
            "anomaly",
        ]
    )
)

print(f"Cleaned after anomaly removal: {len(df_cleaned)}")

<a id="section-8"></a>## 8. Additional CleaningStandardize string columns and ensure consistent casing.

In [ ]:
_STR_COLS = (
    df_cleaned
    .select_dtypes(include=["object"])
    .columns
    .tolist()
)
for _c in _STR_COLS:
    df_cleaned[_c] = df_cleaned[_c].astype(str).str.strip()

_UPPER_COLS = [
    "town",
    "flat_type",
    "flat_model",
    "storey_range",
    "block",
    "street_name",
]
for _c in _UPPER_COLS:
    if _c in df_cleaned.columns:
        df_cleaned[_c] = df_cleaned[_c].str.upper()

_NUM_COLS = [
    "resale_price",
    "floor_area_sqm",
    "lease_commence_date",
]
for _c in _NUM_COLS:
    if _c in df_cleaned.columns:
        df_cleaned[_c] = pd.to_numeric(
            df_cleaned[_c],
            errors="coerce",
        )

print(f"Records after cleaning: {len(df_cleaned)}")
print("Sample:")
print(
    df_cleaned[
        [
            "month",
            "town",
            "flat_type",
            "block",
            "street_name",
            "resale_price",
        ]
    ]
    .head()
)

<a id="section-9"></a>## 9. Resale IdentifierFormat: S + BlockDigits + AvgPriceDigits + MonthDigits + TownChar| Segment | Source ||---------|--------|| S | Fixed first character || BlockDigits | First 3 digits of block, stripped, zero-padded || AvgPriceDigits | 1st and 2nd digit of average resale price per group || MonthDigits | 2-digit month of the entry || TownChar | First character of town |

In [ ]:
df_transformed = df_cleaned.copy()

df_transformed["month_dt"] = pd.to_datetime(
    df_transformed["month"],
    format="%Y-%m",
    errors="coerce",
)
df_transformed["year_month"] = df_transformed["month"]
df_transformed["month_num"] = df_transformed["month_dt"].dt.month


def extract_block_digits(block_val):
    """
    Extract the first 3 digits from a block string.
    """
    _digits = "".join(_ch for _ch in str(block_val) if _ch.isdigit())
    return _digits[:3].zfill(3)


df_transformed["block_digits"] = (
    df_transformed["block"]
    .apply(extract_block_digits)
)

# Average resale price per (year-month, town, flat_type)
_avg_price = (
    df_transformed
    .groupby(["year_month", "town", "flat_type"])
    ["resale_price"]
    .mean()
    .reset_index()
)
_avg_price["avg_price_str"] = (
    _avg_price["resale_price"]
    .apply(lambda _x: str(int(round(_x))))
)
_avg_price["price_digits"] = (
    _avg_price["avg_price_str"]
    .apply(lambda _s: _s[:2].zfill(2))
)

df_transformed = df_transformed.merge(
    _avg_price[
        [
            "year_month",
            "town",
            "flat_type",
            "price_digits",
        ]
    ],
    on=["year_month", "town", "flat_type"],
    how="left",
)

df_transformed["month_digits"] = (
    df_transformed["month_num"]
    .apply(
        lambda _m: f"{int(_m):02d}" if pd.notna(_m) else "00"
    )
)

df_transformed["town_char"] = df_transformed["town"].str[0]


def assemble_identifier(row):
    """
    Assemble the Resale Identifier from its components.
    """
    return (
        f"S{row['block_digits']}"
        f"{row['price_digits']}"
        f"{row['month_digits']}"
        f"{row['town_char']}"
    )


df_transformed["resale_identifier"] = (
    df_transformed
    .apply(assemble_identifier, axis=1)
)

# Drop helper columns
_DROP_COLS = [
    "month_dt",
    "year_month",
    "month_num",
    "block_digits",
    "price_digits",
    "month_digits",
    "town_char",
]
df_transformed = df_transformed.drop(
    columns=_DROP_COLS,
)

print(f"Resale Identifier created for {len(df_transformed)} records.")
print("Sample:")
print(
    df_transformed[
        [
            "month",
            "town",
            "flat_type",
            "block",
            "resale_price",
            "resale_identifier",
        ]
    ]
    .head(10)
)

### 9.1 Identifier Collision Diagnosis

`resale_identifier` is deliberately lossy: `AvgPriceDigits` is only the first 2 digits of the *group* average price for `(year_month, town, flat_type)` (identical for every row in that group), `MonthDigits` ignores the year, and `TownChar` is just the town's first letter. Two different sales can easily land on the same code.

By this point in the pipeline (after Section 6's composite-key dedup on every other column), every remaining record is already known to be a genuinely distinct sale. So a collision here cannot be a real duplicate transaction: it can only be the identifier failing to distinguish two different sales. The cell below quantifies how often that happens and confirms colliding rows differ on fields the composite key already treats as distinguishing.

In [ ]:
_id_counts = df_transformed["resale_identifier"].value_counts()
_colliding_ids = _id_counts[_id_counts > 1].index
_collision_mask = df_transformed["resale_identifier"].isin(_colliding_ids)

_n_total = len(df_transformed)
_n_unique_ids = df_transformed["resale_identifier"].nunique()
_n_collision_groups = len(_colliding_ids)
_n_collision_rows = int(_collision_mask.sum())

print("Resale Identifier collision stats:")
print(f"  Total records:              {_n_total}")
print(f"  Distinct identifiers:       {_n_unique_ids}")
print(f"  Colliding identifier groups: {_n_collision_groups}")
print(
    f"  Records affected by a collision: {_n_collision_rows} "
    f"({_n_collision_rows / _n_total:.1%})"
)

_example_id = _colliding_ids[0]
print(f"\nExample collision group: {_example_id}")
print(
    df_transformed[df_transformed["resale_identifier"] == _example_id][
        [
            "month",
            "town",
            "flat_type",
            "block",
            "street_name",
            "storey_range",
            "floor_area_sqm",
            "lease_commence_date",
            "resale_price",
            "resale_identifier",
        ]
    ]
)

# If these were real duplicate transactions, they would already have been
# removed by Section 6's composite-key dedup. Confirm they are not exact
# duplicates on every other field -- i.e. they are distinct sales that the
# identifier is simply too coarse to tell apart.
_other_cols = [
    c for c in df_transformed.columns if c != "resale_identifier"
]
_exact_dupes_within_collisions = int(
    df_transformed[_collision_mask]
    .duplicated(subset=_other_cols)
    .sum()
)
print(
    f"\nOf the {_n_collision_rows} colliding records, "
    f"{_exact_dupes_within_collisions} are exact duplicates on every other "
    f"field. The remaining {_n_collision_rows - _exact_dupes_within_collisions} "
    f"are genuinely distinct transactions sharing a coincidentally identical "
    f"identifier."
)

### 9.2 Disambiguating Identifier Collisions

Because collisions here are an artifact of the identifier's limited encoding rather than real duplicate sales, dropping the lower-priced record (as an earlier version of this notebook did) silently discarded genuine transactions (roughly a quarter of all cleaned records in a full run).

The fix: keep every record. Within each colliding identifier group, append a numeric suffix (`-1`, `-2`, ...) ordered by descending price so the base 9-character code from the spec is preserved wherever it is already unique, and every record, including every colliding one, ends up with its own unique `resale_identifier`. No records are discarded at this stage.

In [ ]:
df_transformed = df_transformed.sort_values(
    by=["resale_identifier", "resale_price"],
    ascending=[True, False],
).reset_index(drop=True)

_group_sizes = df_transformed.groupby("resale_identifier")[
    "resale_identifier"
].transform("size")
_collision_rank = (
    df_transformed.groupby("resale_identifier").cumcount() + 1
)
_was_collision = _group_sizes > 1

df_transformed["resale_identifier"] = np.where(
    _was_collision,
    df_transformed["resale_identifier"] + "-" + _collision_rank.astype(str),
    df_transformed["resale_identifier"],
)

# No records are discarded here -- collisions are disambiguated, not dropped.
df_id_dup_failed = pd.DataFrame()
N_ID_COLLISIONS_DISAMBIGUATED = int(_was_collision.sum())

print(f"Identifier collisions disambiguated: {N_ID_COLLISIONS_DISAMBIGUATED} records")
print(f"Records retained (none discarded): {len(df_transformed)}")
print(
    f"Unique identifiers: {df_transformed['resale_identifier'].nunique()} "
    f"/ {len(df_transformed)} records"
)

<a id="section-10"></a>## 10. Hash Identifier**Algorithm chosen: SHA-256 (full 64-character hex digest)**- Cryptographically secure, irreversible one-way hash function.- Full digest preserves uniqueness (collision probability ~1 in 2^256).- Deterministic: same input always yields same output.- No salt is added because the identifier contains no sensitive data.

In [ ]:
def hash_sha256(value):
    """
    Compute a SHA-256 hex digest of the input value.
    """
    if pd.isna(value):
        return None
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()

df_hashed = df_transformed.copy()
df_hashed["resale_identifier_hash"] = (
    df_hashed["resale_identifier"]
    .apply(hash_sha256)
)

print("Hashed identifier column created.")
print("Sample:")
print(
    df_hashed[
        [
            "resale_identifier",
            "resale_identifier_hash",
        ]
    ]
    .head()
)

_unique_ids = df_hashed["resale_identifier"].nunique()
_unique_hashes = df_hashed["resale_identifier_hash"].nunique()
print(f"Unique identifiers: {_unique_ids}")
print(f"Unique hashes:      {_unique_hashes}")
print(f"Uniqueness preserved: {_unique_ids == _unique_hashes}")

<a id="section-11"></a>
## 11. Assemble Failed Dataset

Combine every record removed at any stage: validation failures, composite-key duplicates, and anomalies. (Resale Identifier collisions are disambiguated in Section 9.2, not discarded, so `df_id_dup_failed` is always empty and contributes no rows here.)

In [ ]:
_FAILED_PARTS = [
    df_failed_validation,
    df_dup_discarded,
    df_anomaly_failed,
    df_id_dup_failed,
]

_ALL_FAILED_COLS = set()
for _d in _FAILED_PARTS:
    if not _d.empty:
        _ALL_FAILED_COLS.update(_d.columns)

_ALL_FAILED_COLS = sorted(_ALL_FAILED_COLS)

df_failed = pd.DataFrame(columns=_ALL_FAILED_COLS)
for _d in _FAILED_PARTS:
    if not _d.empty:
        _aligned = _d.reindex(columns=_ALL_FAILED_COLS)
        df_failed = pd.concat(
            [df_failed, _aligned],
            ignore_index=True,
        )

print(f"Total failed records: {len(df_failed)}")
print("Failure reasons:")
print(df_failed["validation_error"].value_counts())

<a id="section-12"></a>## 12. Save OutputsProduce the five required output groups:| Output | Description ||--------|-------------|| raw/ | Raw data files as-is || cleaned.csv | Records passing all data quality requirements || transformed.csv | Cleaned data + unhashed resale_identifier || failed.csv | All removed records with error reasons || hashed.csv | Cleaned data + hashed identifier column |

In [ ]:
df_cleaned.to_csv(
    PROCESSED_DIR / "cleaned.csv",
    index=False,
)
df_transformed.to_csv(
    PROCESSED_DIR / "transformed.csv",
    index=False,
)
df_hashed.to_csv(
    PROCESSED_DIR / "hashed.csv",
    index=False,
)
df_failed.to_csv(
    PROCESSED_DIR / "failed.csv",
    index=False,
)

_N_RAW = len(list(RAW_OUT_DIR.glob("*.csv")))
print("Output files saved:")
print(f"  - raw/            ({_N_RAW} files)")
print(f"  - cleaned.csv     ({len(df_cleaned)} records)")
print(f"  - transformed.csv ({len(df_transformed)} records)")
print(f"  - hashed.csv      ({len(df_hashed)} records)")
print(f"  - failed.csv      ({len(df_failed)} records)")
print(f"All saved to: {PROCESSED_DIR}")

<a id="section-13"></a>## 13. Pipeline SummaryEnd-to-end pipeline statistics.

In [ ]:
print("=" * 60)
print("       HDB RESALE FLAT PRICE ETL PIPELINE SUMMARY")
print("=" * 60)
print(f"Total input records (combined):    {len(combined_df)}")
print(f"Failed validation:                 {len(df_failed_validation)}")
print(f"Duplicate composite-key discards:  {len(df_dup_discarded)}")
print(f"Anomalous price discards:          {len(df_anomaly_failed)}")
print(f"Identifier collisions disambiguated (not discarded): {N_ID_COLLISIONS_DISAMBIGUATED}")
print(f"Final cleaned records:             {len(df_cleaned)}")
print(f"Final transformed records:         {len(df_transformed)}")
print(f"Final hashed records:              {len(df_hashed)}")
print(f"Total failed records:              {len(df_failed)}")
print("=" * 60)